#  Swasthya: Distributor Demand Aggregation

**What this notebook does:**  
Aggregates pharmacy-level demand forecasts (from Feature 1's LightGBM model) across all pharmacies assigned to each distributor — giving distributors a consolidated view of expected incoming orders before they arrive.

**Who sees this feature:** Distributor dashboard only.

**Key design decision — no new model is trained.**  
Feature 1's LightGBM already predicts per-pharmacy per-SKU demand.  
Feature 3 sums those predictions per distributor.  
One model, two views of its output.

**Output:**
- `aggregated_demand.csv` — 7-day demand per distributor per SKU
- `distributor_risk.csv` — stock risk assessment
- `dist_metadata.json` — config for FastAPI

---
```
STEP 1  → Install & import
STEP 2  → Load all data + Feature 1 trained model
STEP 3  → Map pharmacies to distributors
STEP 4  → Run Feature 1 forecasts for every pharmacy
STEP 5  → Aggregate forecasts per distributor
STEP 6  → Load distributor current stock
STEP 7  → Compare demand vs stock → flag risk
STEP 8  → Week-over-week trend analysis
STEP 9  → Gemini supply insight generation
STEP 10 → Full distributor dashboard simulation
STEP 11 → Save all output files
STEP 12 → FastAPI-ready inference function
```

##  Install & Import

In [ ]:
!pip install lightgbm scikit-learn pandas numpy matplotlib seaborn google-generativeai --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import joblib, json, warnings, os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
os.makedirs('/content/ML/models', exist_ok=True)
os.makedirs('/content/ML/outputs', exist_ok=True)
print('✅ All packages imported')

## Load All Data + Feature 1 Model

In [ ]:
from google.colab import files
import zipfile

print('Step 1: Upload swasthya_dataset.zip')
uploaded = files.upload()
with zipfile.ZipFile('swasthya_dataset.zip', 'r') as z:
    z.extractall('/content/')
print('✅ Dataset extracted')

In [ ]:
print('Step 2: Upload feature1_models.zip (downloaded from Feature 1 notebook)')
uploaded2 = files.upload()
with zipfile.ZipFile('feature1_models.zip', 'r') as z:
    z.extractall('/content/ML/models/')
print('✅ Feature 1 models extracted')
for f in sorted(os.listdir('/content/ML/models/')):
    print(f'  {f}')

In [ ]:
sales_df    = pd.read_csv('/content/swasthya_data/daily_sales.csv',       parse_dates=['date'])
dist_stock  = pd.read_csv('/content/swasthya_data/distributor_stock.csv', parse_dates=['snapshot_date'])
pharmacy_df = pd.read_csv('/content/swasthya_data/master_pharmacies.csv')
dist_df     = pd.read_csv('/content/swasthya_data/master_distributors.csv')
products_df = pd.read_csv('/content/swasthya_data/master_products.csv')
orders_df   = pd.read_csv('/content/swasthya_data/pharmacy_orders.csv',   parse_dates=['order_date'])

lgb_model   = joblib.load('/content/ML/models/demand_forecaster.pkl')
f1_meta     = json.load(open('/content/ML/models/model_metadata.json'))
FEATURE_COLS= f1_meta['feature_cols']

print('='*55)
print('  FEATURE 3 — DATA + MODEL LOADED')
print('='*55)
print(f'  Sales rows     : {len(sales_df):,}')
print(f'  Dist stock rows: {len(dist_stock):,}')
print(f'  Pharmacies     : {len(pharmacy_df)}')
print(f'  Distributors   : {len(dist_df)}')
print(f'  LightGBM model : loaded (F1 MAPE = {f1_meta["metrics"]["MAPE"]}%)')
print('='*55)

print('\nDistributor → Pharmacy mapping:')
for _, d in dist_df.iterrows():
    phs = pharmacy_df[pharmacy_df['dist_id']==d['dist_id']]['name'].tolist()
    print(f'  {d["name"][:38]:38s} ({len(phs)} pharmacies)')

## Map Pharmacies to Distributors

In [ ]:
dist_pharmacy_map = (
    pharmacy_df
    .merge(dist_df, left_on='dist_id', right_on='dist_id', how='left')
    .rename(columns={'name_x':'pharmacy_name','name_y':'distributor_name',
                 'dist_id':'distributor_id', 'city_x':'city'})   # ← add this
    [['pharmacy_id','pharmacy_name','city','tier',
      'pop_factor','distributor_id','distributor_name',
      'reliability','lead_time_days']]
)

print('Pharmacy → Distributor assignment:\n')
for dist_id in dist_pharmacy_map['distributor_id'].unique():
    sub  = dist_pharmacy_map[dist_pharmacy_map['distributor_id']==dist_id]
    dname= sub['distributor_name'].iloc[0]
    print(f'  📦 {dname}')
    for _, r in sub.iterrows():
        print(f'     └─ {r["pharmacy_name"]} ({r["city"]}, Tier {r["tier"]})')
    print()

print(f'✅ {len(dist_pharmacy_map)} assignments mapped')

## Run Feature 1 Forecasts for Every Pharmacy

In [ ]:
def engineer_features(df):
    """Identical feature pipeline from Feature 1 notebook."""
    df = df.sort_values(['pharmacy_id','sku_id','date']).copy()
    grp = df.groupby(['pharmacy_id','sku_id'])['quantity_sold']

    df['lag_1']   = grp.shift(1)
    df['lag_7']   = grp.shift(7)
    df['lag_14']  = grp.shift(14)
    df['lag_28']  = grp.shift(28)
    df['lag_91']  = grp.shift(91)
    df['lag_365'] = grp.shift(365)

    df['roll_7_mean']  = grp.transform(lambda x: x.shift(1).rolling(7,  min_periods=3).mean())
    df['roll_7_std']   = grp.transform(lambda x: x.shift(1).rolling(7,  min_periods=3).std())
    df['roll_14_mean'] = grp.transform(lambda x: x.shift(1).rolling(14, min_periods=7).mean())
    df['roll_30_mean'] = grp.transform(lambda x: x.shift(1).rolling(30, min_periods=14).mean())
    df['roll_60_mean'] = grp.transform(lambda x: x.shift(1).rolling(60, min_periods=30).mean())
    df['roll_90_mean'] = grp.transform(lambda x: x.shift(1).rolling(90, min_periods=45).mean())

    df['trend_7_vs_30']  = df['roll_7_mean']  / (df['roll_30_mean']  + 1e-6)
    df['trend_14_vs_60'] = df['roll_14_mean'] / (df['roll_60_mean']  + 1e-6)
    df['trend_30_vs_90'] = df['roll_30_mean'] / (df['roll_90_mean']  + 1e-6)

    df['sin_dow']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['cos_dow']   = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['sin_doy']   = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['cos_doy']   = np.cos(2 * np.pi * df['day_of_year'] / 365)
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

    df['sku_code']  = df['sku_id'].str.extract(r'(\d+)').astype(int)
    df['ph_code']   = df['pharmacy_id'].str.extract(r'(\d+)').astype(int)
    df['city_code'] = df['city'].astype('category').cat.codes
    df['cat_code']  = df['category'].astype('category').cat.codes

    return df.dropna(subset=['lag_1','lag_7','roll_7_mean','roll_30_mean'])


print('⏳ Engineering features...')
feat_df = engineer_features(sales_df)
print(f'✅ {len(feat_df):,} rows ready')

In [ ]:
def predict_pharmacy_7d(pharmacy_id, sku_id, feat_df, model, FEATURE_COLS):
    """Predict 7-day demand for one pharmacy+SKU. Reused from Feature 1."""
    subset = feat_df[
        (feat_df['pharmacy_id']==pharmacy_id) &
        (feat_df['sku_id']==sku_id)
    ].sort_values('date')

    if len(subset) == 0:
        return None

    latest     = subset.iloc[-1]
    features   = latest[FEATURE_COLS].values.reshape(1, -1)
    daily_pred = float(np.maximum(model.predict(features)[0], 0))

    roll_std = latest.get('roll_7_std', daily_pred * 0.2)
    if pd.isna(roll_std):
        roll_std = daily_pred * 0.2

    return {
        'pharmacy_id':  pharmacy_id,
        'sku_id':       sku_id,
        'product_name': latest['product_name'],
        'category':     latest['category'],
        'unit_price':   int(latest['unit_price']),
        'daily_pred':   round(daily_pred, 3),
        'forecast_7d':  round(daily_pred * 7, 1),
        'ci_lower_7d':  round(max(0, (daily_pred - 1.5*roll_std)*7), 1),
        'ci_upper_7d':  round((daily_pred + 1.5*roll_std)*7, 1),
    }


print('⏳ Running Feature 1 model for all 15 pharmacies × 7 products...')
all_ph_forecasts = []
for ph_id in sales_df['pharmacy_id'].unique():
    for sku_id in sales_df['sku_id'].unique():
        r = predict_pharmacy_7d(ph_id, sku_id, feat_df, lgb_model, FEATURE_COLS)
        if r:
            all_ph_forecasts.append(r)

ph_forecasts_df = pd.DataFrame(all_ph_forecasts)
ph_forecasts_df = ph_forecasts_df.merge(
    dist_pharmacy_map[['pharmacy_id','pharmacy_name','city','tier',
                       'distributor_id','distributor_name','lead_time_days']],
    on='pharmacy_id', how='left'
)

print(f'✅ {len(ph_forecasts_df)} pharmacy-level forecasts generated')
print()
print('Sample (PH_01 forecasts):')
ph_forecasts_df[
    ph_forecasts_df['pharmacy_id']=='PH_01'
][['product_name','forecast_7d','distributor_name']].to_string(index=False)

## Aggregate Forecasts Per Distributor

In [ ]:
dist_aggregated = (
    ph_forecasts_df
    .groupby(['distributor_id','distributor_name','sku_id','product_name',
              'category','unit_price'])
    .agg(
        n_pharmacies     = ('pharmacy_id',  'count'),
        total_forecast_7d= ('forecast_7d',  'sum'),
        total_ci_lower   = ('ci_lower_7d',  'sum'),
        total_ci_upper   = ('ci_upper_7d',  'sum'),
        avg_daily_per_ph = ('daily_pred',   'mean'),
        max_ph_demand    = ('forecast_7d',  'max'),
        min_ph_demand    = ('forecast_7d',  'min'),
    )
    .reset_index()
)

dist_aggregated['total_forecast_7d'] = dist_aggregated['total_forecast_7d'].round(0).astype(int)
dist_aggregated['total_ci_lower']    = dist_aggregated['total_ci_lower'].round(0).astype(int)
dist_aggregated['total_ci_upper']    = dist_aggregated['total_ci_upper'].round(0).astype(int)
dist_aggregated['total_order_value'] = dist_aggregated['total_forecast_7d'] * dist_aggregated['unit_price']

print('='*55)
print('  AGGREGATED DEMAND — ALL DISTRIBUTORS')
print('='*55)
for did in dist_aggregated['distributor_id'].unique():
    sub   = dist_aggregated[dist_aggregated['distributor_id']==did]
    name  = sub['distributor_name'].iloc[0]
    total = sub['total_forecast_7d'].sum()
    value = sub['total_order_value'].sum()
    print(f'  {name}')
    print(f'  Units expected: {total:,}  |  Value: ₹{value:,.0f}\n')

dist_aggregated.head(7)

## Load Distributor Current Stock

In [ ]:
latest_dist_stock = (
    dist_stock
    .sort_values('snapshot_date')
    .groupby(['distributor_id','sku_id'])
    .last()
    .reset_index()
    [['distributor_id','sku_id','product_name','current_stock',
      'reorder_level','max_capacity','is_low_stock','is_out_of_stock','snapshot_date']]
)

print(f'Latest stock snapshot date: {latest_dist_stock["snapshot_date"].max().date()}')
print(f'Rows: {len(latest_dist_stock)}')
print()

stock_summary = (
    latest_dist_stock
    .merge(dist_df[['dist_id','name']], left_on='distributor_id', right_on='dist_id')
    .groupby('name')
    .agg(total_units=('current_stock','sum'),
         low_stock_items=('is_low_stock','sum'),
         out_of_stock=('is_out_of_stock','sum'))
)
print('Current Stock Summary:')
print(stock_summary.to_string())

## Compare Demand vs Stock → Flag Risk

In [ ]:
risk_df = dist_aggregated.merge(
    latest_dist_stock[['distributor_id','sku_id','current_stock',
                       'reorder_level','max_capacity','is_low_stock','is_out_of_stock']],
    on=['distributor_id','sku_id'], how='left'
)

risk_df['daily_demand_total'] = risk_df['total_forecast_7d'] / 7
risk_df['stock_cover_days']   = (risk_df['current_stock'] / risk_df['daily_demand_total'].clip(lower=0.1)).round(1)
risk_df['stock_gap']          = (risk_df['total_forecast_7d'] - risk_df['current_stock']).clip(lower=0).astype(int)
risk_df['shortfall_value']    = risk_df['stock_gap'] * risk_df['unit_price']
risk_df['coverage_ratio']     = (risk_df['current_stock'] / risk_df['total_forecast_7d'].clip(lower=1)).round(3)

def assign_risk(row):
    if row['is_out_of_stock']:        return 'CRITICAL — Out of Stock'
    if row['stock_cover_days'] <= 2:  return 'CRITICAL — < 2 days'
    if row['coverage_ratio'] < 0.5:   return 'HIGH — < 50% coverage'
    if row['coverage_ratio'] < 0.8:   return 'MEDIUM — < 80% coverage'
    if row['stock_gap'] > 0:          return 'LOW — Minor shortfall'
    return 'OK — Sufficient Stock'

risk_df['risk_level']    = risk_df.apply(assign_risk, axis=1)
risk_df['needs_restock'] = risk_df['stock_gap'] > 0
risk_df['restock_qty']   = risk_df['stock_gap']
risk_df['restock_value'] = risk_df['restock_qty'] * risk_df['unit_price']

print('='*55)
print('  DISTRIBUTOR STOCK RISK ASSESSMENT')
print('='*55)
for level, count in risk_df['risk_level'].value_counts().items():
    print(f'  {level:35s}: {count} SKUs')
print()
print(f'  Total shortfall value: ₹{risk_df["shortfall_value"].sum():,.0f}')
print('='*55)
print()
print('Riskiest SKUs:')
risk_df[risk_df['needs_restock']].sort_values('coverage_ratio').head(8)[
    ['distributor_name','product_name','current_stock','total_forecast_7d',
     'stock_gap','coverage_ratio','risk_level']
].to_string(index=False)

## Week-over-Week Trend Analysis

In [ ]:
# Last week actual orders per distributor+SKU
last_week_cutoff = orders_df['order_date'].max() - pd.Timedelta(days=7)
last_week_orders = (
    orders_df[
        (orders_df['order_date'] >= last_week_cutoff) &
        (orders_df['status'] != 'cancelled')
    ]
    .groupby(['distributor_id','sku_id'])['order_qty']
    .sum()
    .reset_index()
    .rename(columns={'order_qty':'last_week_actual'})
)

trend_df = risk_df.merge(last_week_orders, on=['distributor_id','sku_id'], how='left')
trend_df['last_week_actual'] = trend_df['last_week_actual'].fillna(0)
trend_df['wow_change_pct']   = (
    (trend_df['total_forecast_7d'] - trend_df['last_week_actual']) /
    trend_df['last_week_actual'].clip(lower=1) * 100
).round(1)
trend_df['trend_direction'] = trend_df['wow_change_pct'].apply(
    lambda x: '📈 Rising' if x > 10 else ('📉 Falling' if x < -10 else '➡️  Stable')
)

# ── Charts ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('📦 Distributor Demand Aggregation — Overview', fontsize=14, fontweight='bold')

# 1. Forecast vs last week per distributor
dist_totals = trend_df.groupby('distributor_name').agg(
    forecast=('total_forecast_7d','sum'), last_actual=('last_week_actual','sum')
).reset_index()
short_dist = [n.split(' ')[0] for n in dist_totals['distributor_name']]
x = np.arange(len(dist_totals))
axes[0,0].bar(x-0.2, dist_totals['last_actual'], 0.4, color='#78909C', label='Last Week Actual', alpha=0.85)
axes[0,0].bar(x+0.2, dist_totals['forecast'],    0.4, color='#7B1FA2', label='This Week Forecast', alpha=0.85)
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(short_dist, fontsize=9)
axes[0,0].set_title('Last Week vs This Week Forecast\n(Total Units per Distributor)')
axes[0,0].set_ylabel('Units')
axes[0,0].legend()

# 2. Risk level breakdown
risk_color_map = {
    'OK — Sufficient Stock':    '#43A047',
    'LOW — Minor shortfall':    '#FB8C00',
    'MEDIUM — < 80% coverage':  '#F4511E',
    'HIGH — < 50% coverage':    '#C62828',
    'CRITICAL — < 2 days':      '#4A148C',
    'CRITICAL — Out of Stock':  '#1A237E',
}
rc = risk_df['risk_level'].value_counts()
axes[0,1].barh(rc.index, rc.values,
               color=[risk_color_map.get(r,'gray') for r in rc.index])
axes[0,1].set_title('Risk Level Distribution\n(All Distributors × All SKUs)')
axes[0,1].set_xlabel('Number of SKUs')

# 3. WoW trend per product
wow_by_prod = trend_df.groupby('product_name')['wow_change_pct'].mean().sort_values()
bar_colors  = ['#E53935' if v < 0 else '#43A047' for v in wow_by_prod.values]
short_prods = [p.replace(' with ',' w/ ')[:22] for p in wow_by_prod.index]
axes[1,0].barh(short_prods, wow_by_prod.values, color=bar_colors, alpha=0.85)
axes[1,0].axvline(0, color='black', linewidth=1.5)
axes[1,0].set_title('Week-over-Week Demand Trend (avg across distributors)')
axes[1,0].set_xlabel('% Change')

# 4. Coverage ratio heatmap
coverage_pivot = risk_df.pivot_table(
    values='coverage_ratio', index='distributor_id',
    columns='sku_id', aggfunc='mean'
)
sns.heatmap(
    coverage_pivot, ax=axes[1,1], cmap='RdYlGn', vmin=0, vmax=2,
    annot=True, fmt='.1f', linewidths=0.5
)
axes[1,1].set_title('Stock Coverage Heatmap\n(Current Stock / 7-day Forecast, >1 = OK)')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('/content/ML/models/distributor_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Overview chart saved')

## Gemini Supply Insight Generation

In [ ]:
import google.generativeai as genai

# Get free key from: https://makersuite.google.com/app/apikey
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'  # ← paste your key

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')
print('✅ Gemini configured')

In [ ]:
def generate_distributor_insight(distributor_id, trend_df, dist_df):
    """
    Generate a plain-language supply briefing for a distributor.
    Identifies riskiest SKUs and biggest demand trend changes.
    """
    subset    = trend_df[trend_df['distributor_id'] == distributor_id].copy()
    dist_info = dist_df[dist_df['dist_id'] == distributor_id].iloc[0]

    at_risk  = subset[subset['needs_restock']].sort_values('coverage_ratio')
    rising   = subset[subset['wow_change_pct'] > 10].sort_values('wow_change_pct', ascending=False)
    total_exp= subset['total_forecast_7d'].sum()
    total_gap= subset['stock_gap'].sum()

    risk_lines = [
        f"  - {r['product_name']}: stock={int(r['current_stock'])}, "
        f"expected={int(r['total_forecast_7d'])}, shortfall={int(r['stock_gap'])}, "
        f"coverage={r['coverage_ratio']:.0%}"
        for _, r in at_risk.head(4).iterrows()
    ]
    trend_lines = [
        f"  - {r['product_name']}: +{r['wow_change_pct']:.0f}% vs last week"
        for _, r in rising.head(3).iterrows()
    ]

    prompt = f"""You are a supply chain assistant for Swasthya, an Indian hygiene product B2B platform.
Write a concise supply briefing for the distributor. Maximum 3 sentences.
Mention the most urgent stock risk first, then the top demand trend. No markdown. No bullet points.

Distributor : {dist_info['name']}, {dist_info['city']}
Lead time   : {dist_info['lead_time_days']} day(s)
Total expected (7d): {total_exp:,} units | Total shortfall: {total_gap:,} units

Stock at risk:
{chr(10).join(risk_lines) if risk_lines else '  None'}

Rising demand:
{chr(10).join(trend_lines) if trend_lines else '  No significant rises'}"""

    try:
        return gemini.generate_content(prompt).text.strip()
    except Exception:
        if risk_lines:
            return (
                f"Stock shortfall of {total_gap:,} units detected across {len(at_risk)} products. "
                f"Order from admin immediately — lead time is {dist_info['lead_time_days']} day(s)."
            )
        return f"All stocks sufficient. {total_exp:,} units expected across 7 products next week."


print('💬 GEMINI DISTRIBUTOR BRIEFINGS\n')
print('─'*62)

dist_insights = {}
for dist_id in dist_df['dist_id'].unique():
    insight   = generate_distributor_insight(dist_id, trend_df, dist_df)
    dist_insights[dist_id] = insight
    dname  = dist_df[dist_df['dist_id']==dist_id]['name'].values[0]
    dcity  = dist_df[dist_df['dist_id']==dist_id]['city'].values[0]
    sub    = trend_df[trend_df['distributor_id']==dist_id]
    n_risk = sub['needs_restock'].sum()

    print(f'\n📦 {dname} ({dcity})')
    print(f'   At-risk SKUs: {n_risk}/7 | Expected: {sub["total_forecast_7d"].sum():,} units')
    print(f'   🤖 AI: {insight}')
    print('─'*62)

## Full Distributor Dashboard Simulation

In [ ]:
def print_distributor_dashboard(distributor_id, trend_df, dist_df, ph_forecasts_df, dist_insights):
    """Simulates what the distributor sees on their Expo dashboard."""
    dist_info = dist_df[dist_df['dist_id']==distributor_id].iloc[0]
    subset    = trend_df[trend_df['distributor_id']==distributor_id].sort_values('coverage_ratio')

    risk_icons = {
        'CRITICAL — Out of Stock':   '🔴',
        'CRITICAL — < 2 days':       '🔴',
        'HIGH — < 50% coverage':     '🟠',
        'MEDIUM — < 80% coverage':   '🟡',
        'LOW — Minor shortfall':     '🔵',
        'OK — Sufficient Stock':     '✅',
    }

    n_ph       = len(ph_forecasts_df[ph_forecasts_df['distributor_id']==distributor_id]['pharmacy_id'].unique())
    total_exp  = subset['total_forecast_7d'].sum()
    total_stock= subset['current_stock'].sum()
    total_gap  = subset['stock_gap'].sum()
    n_risk     = subset['needs_restock'].sum()

    print(f'\n{"="*65}')
    print(f'  📦 DISTRIBUTOR DASHBOARD')
    print(f'  {dist_info["name"]}')
    print(f'  {dist_info["city"]} | Lead time: {dist_info["lead_time_days"]}d | Reliability: {dist_info["reliability"]*100:.0f}%')
    print(f'{"="*65}')
    print(f'  Assigned Pharmacies : {n_ph}')
    print(f'  Expected (7d)       : {total_exp:,} units')
    print(f'  Current Stock       : {total_stock:,} units')
    print(f'  Total Shortfall     : {total_gap:,} units')
    print(f'  SKUs at Risk        : {n_risk}/7')
    print()
    print(f'  {"Product":<35} {"Expected":>9} {"Stock":>7} {"Gap":>6} {"WoW%":>6}  Risk')
    print(f'  {"─"*65}')
    for _, row in subset.iterrows():
        icon = risk_icons.get(row['risk_level'], '❓')
        wow  = f"+{row['wow_change_pct']:.0f}%" if row['wow_change_pct'] > 0 else f"{row['wow_change_pct']:.0f}%"
        print(
            f'  {row["product_name"][:35]:<35} '
            f'{int(row["total_forecast_7d"]):>9,} '
            f'{int(row["current_stock"]):>7,} '
            f'{int(row["stock_gap"]):>6,} '
            f'{wow:>6}  '
            f'{icon} {row["risk_level"].split(" — ")[0]}'
        )
    print()
    print(f'  💬 AI Briefing: {dist_insights.get(distributor_id, "No insight.")}')
    print(f'{"="*65}')


# Print all 5 distributors
for dist_id in dist_df['dist_id'].unique():
    print_distributor_dashboard(dist_id, trend_df, dist_df, ph_forecasts_df, dist_insights)

In [ ]:
# ── Detailed chart for one distributor ───────────────────────────────────────
DEMO_DIST = 'DIST_01'
demo_data = trend_df[trend_df['distributor_id']==DEMO_DIST].copy()
demo_info = dist_df[dist_df['dist_id']==DEMO_DIST].iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    f'📦 {demo_info["name"]} ({demo_info["city"]}) — Demand Dashboard',
    fontsize=13, fontweight='bold'
)

short_prods = [p.replace(' with ',' w/ ')[:20] for p in demo_data['product_name']]
x = np.arange(len(demo_data))

# 1. Stock vs forecast
axes[0].bar(x-0.2, demo_data['current_stock'],     0.4, color='#78909C', label='Current Stock', alpha=0.85)
axes[0].bar(x+0.2, demo_data['total_forecast_7d'], 0.4, color='#7B1FA2', label='Expected 7d',  alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_prods, rotation=30, ha='right', fontsize=8)
axes[0].set_title('Current Stock vs 7-Day Forecast')
axes[0].set_ylabel('Units')
axes[0].legend()

# 2. Coverage ratio
cov_colors = ['#C62828' if v<0.5 else '#FB8C00' if v<0.8 else '#FDD835' if v<1 else '#43A047'
              for v in demo_data['coverage_ratio']]
axes[1].bar(short_prods, demo_data['coverage_ratio'], color=cov_colors, alpha=0.9)
axes[1].axhline(1.0, color='black',  linestyle='--', linewidth=1.5, label='Safe (1.0)')
axes[1].axhline(0.8, color='orange', linestyle=':',  linewidth=1.5, label='Warning (0.8)')
axes[1].set_title('Stock Coverage Ratio')
axes[1].set_ylabel('Ratio')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8)

# 3. WoW change
wow_colors = ['#E53935' if v < 0 else '#43A047' for v in demo_data['wow_change_pct']]
axes[2].bar(short_prods, demo_data['wow_change_pct'], color=wow_colors, alpha=0.85)
axes[2].axhline(0,   color='black', linewidth=1.5)
axes[2].axhline(10,  color='green', linestyle=':', linewidth=1, label='+10%')
axes[2].axhline(-10, color='red',   linestyle=':', linewidth=1, label='-10%')
axes[2].set_title('Week-over-Week Demand Trend (%)')
axes[2].set_ylabel('% Change')
axes[2].tick_params(axis='x', rotation=30)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/content/ML/models/distributor_dashboard_detail.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Detailed chart saved for {demo_info["name"]}')

## Save All Output Files

In [ ]:
trend_df.to_csv('/content/ML/outputs/aggregated_demand.csv', index=False)
risk_df.to_csv('/content/ML/outputs/distributor_risk.csv',   index=False)
ph_forecasts_df.to_csv('/content/ML/outputs/pharmacy_forecasts.csv', index=False)
pd.DataFrame([{'distributor_id':k,'insight':v} for k,v in dist_insights.items()])\
  .to_csv('/content/ML/outputs/distributor_insights.csv', index=False)

metadata = {
    'project':       'Swasthya — Intelligent Pharmacy Network',
    'feature':       'Feature 3 — Distributor Demand Aggregation',
    'model_used':    'Feature 1 LightGBM (demand_forecaster.pkl) — no new model trained',
    'feature_cols':  FEATURE_COLS,
    'forecast_horizon_days': 7,
    'aggregation':   'Sum of pharmacy-level forecasts per distributor per SKU',
    'risk_thresholds': {
        'CRITICAL': 'stock_cover_days <= 2 OR out_of_stock',
        'HIGH':     'coverage_ratio < 0.5',
        'MEDIUM':   'coverage_ratio < 0.8',
        'LOW':      'stock_gap > 0 but coverage_ratio >= 0.8',
        'OK':       'coverage_ratio >= 1.0',
    },
    'trend_thresholds': {
        'Rising':  'wow_change_pct > 10%',
        'Falling': 'wow_change_pct < -10%',
        'Stable':  '-10% to +10%',
    },
    'distributors': dist_df.to_dict('records'),
    'n_pharmacies':  len(pharmacy_df),
    'n_skus':        7,
}
with open('/content/ML/outputs/dist_metadata.json','w') as f:
    json.dump(metadata, f, indent=2)

print('✅ All files saved:')
for fname in sorted(os.listdir('/content/ML/outputs/')):
    size = os.path.getsize(f'/content/ML/outputs/{fname}')
    print(f'  {fname:45s}  {size/1024:.1f} KB')

# Download
import zipfile
from google.colab import files
with zipfile.ZipFile('/content/feature3_outputs.zip','w') as zf:
    for fname in os.listdir('/content/ML/outputs/'):
        zf.write(f'/content/ML/outputs/{fname}', fname)
    for fname in ['distributor_overview.png','distributor_dashboard_detail.png']:
        fpath = f'/content/ML/models/{fname}'
        if os.path.exists(fpath):
            zf.write(fpath, f'charts/{fname}')
print('\n📦 Downloading feature3_outputs.zip...')
files.download('/content/feature3_outputs.zip')

## FastAPI-Ready Inference Function

In [ ]:
def get_distributor_forecast(distributor_id, feat_df, lgb_model, FEATURE_COLS,
                             ph_forecasts_df, dist_stock, dist_df, use_gemini=True):
    """
    Full pipeline for /distributor-forecast endpoint.
    Input:  distributor_id
    Output: aggregated demand + risk + WoW trend + Gemini insight
    """
    dist_ph = ph_forecasts_df[ph_forecasts_df['distributor_id']==distributor_id]
    if len(dist_ph) == 0:
        return {'error': f'No pharmacies assigned to {distributor_id}'}

    # Aggregate by SKU
    agg = (
        dist_ph
        .groupby(['sku_id','product_name','unit_price'])
        .agg(n_pharmacies=('pharmacy_id','count'),
             total_forecast_7d=('forecast_7d','sum'),
             ci_lower=('ci_lower_7d','sum'),
             ci_upper=('ci_upper_7d','sum'))
        .reset_index()
    )
    agg['total_forecast_7d'] = agg['total_forecast_7d'].round(0).astype(int)

    # Get latest distributor stock
    stock = (
        dist_stock[dist_stock['distributor_id']==distributor_id]
        .sort_values('snapshot_date')
        .groupby('sku_id').last().reset_index()
        [['sku_id','current_stock','reorder_level','is_low_stock','is_out_of_stock']]
    )

    result = agg.merge(stock, on='sku_id', how='left')
    result['stock_gap']      = (result['total_forecast_7d'] - result['current_stock']).clip(lower=0).astype(int)
    result['coverage_ratio'] = (result['current_stock'] / result['total_forecast_7d'].clip(lower=1)).round(3)
    result['needs_restock']  = result['stock_gap'] > 0
    result['restock_value']  = result['stock_gap'] * result['unit_price']

    dist_info  = dist_df[dist_df['dist_id']==distributor_id].iloc[0]
    at_risk    = result[result['needs_restock']]['product_name'].tolist()

    ai_insight = None
    if use_gemini:
        prompt = (
            f"Swasthya distributor dashboard. 2-sentence briefing. No markdown.\n"
            f"Distributor: {dist_info['name']}, {dist_info['city']}\n"
            f"Expected incoming: {result['total_forecast_7d'].sum():,} units\n"
            f"Shortfall: {result['stock_gap'].sum():,} units across "
            f"{result['needs_restock'].sum()} products\n"
            f"At risk: {', '.join(at_risk) if at_risk else 'None'}\n"
            f"Lead time: {dist_info['lead_time_days']}d"
        )
        try:
            ai_insight = gemini.generate_content(prompt).text.strip()
        except Exception:
            ai_insight = f"Shortfall of {result['stock_gap'].sum():,} units. Order from admin immediately."

    return {
        'distributor_id':    distributor_id,
        'distributor_name':  dist_info['name'],
        'city':              dist_info['city'],
        'lead_time_days':    int(dist_info['lead_time_days']),
        'n_pharmacies':      int(dist_ph['pharmacy_id'].nunique()),
        'total_expected_7d': int(result['total_forecast_7d'].sum()),
        'total_gap':         int(result['stock_gap'].sum()),
        'n_at_risk_skus':    int(result['needs_restock'].sum()),
        'products':          result.to_dict('records'),
        'ai_insight':        ai_insight,
    }


# Test
print('🔬 Testing inference function for DIST_01...\n')
out = get_distributor_forecast(
    'DIST_01', feat_df, lgb_model, FEATURE_COLS,
    ph_forecasts_df, dist_stock, dist_df, use_gemini=True
)
for k, v in out.items():
    if k != 'products':
        print(f'  {k:25s}: {v}')
print('\nProducts breakdown:')
for p in out['products']:
    icon = '🔴' if p['stock_gap'] > 0 else '✅'
    print(f'  {icon} {p["product_name"][:35]:35s} '
          f'expected={int(p["total_forecast_7d"]):4d} '
          f'stock={int(p["current_stock"]):4d} '
          f'gap={int(p["stock_gap"]):4d}')

In [ ]:
FASTAPI_CODE = '''
# Add to ML/api/main.py alongside Feature 1 and Feature 2 endpoints

dist_df_api    = pd.read_csv("models/master_distributors.csv")
ph_df_api      = pd.read_csv("models/master_pharmacies.csv")
dist_stock_api = pd.read_csv("models/distributor_stock.csv", parse_dates=["snapshot_date"])
# feat_df_cached is already built at startup from the demand forecasting endpoint


@app.get("/distributor-forecast/{distributor_id}")
async def distributor_forecast(distributor_id: str):
    assigned_ph = ph_df_api[ph_df_api["dist_id"]==distributor_id]["pharmacy_id"].tolist()
    if not assigned_ph:
        raise HTTPException(404, detail=f"{distributor_id} not found")

    # Run Feature 1 model for each assigned pharmacy
    all_forecasts = []
    for ph_id in assigned_ph:
        for sku_id in [f"SKU_00{i}" for i in range(1, 8)]:
            r = predict_pharmacy_7d(ph_id, sku_id, feat_df_cached, model, FEATURES)
            if r:
                r["distributor_id"] = distributor_id
                all_forecasts.append(r)

    ph_f = pd.DataFrame(all_forecasts)

    # Aggregate by SKU
    agg = ph_f.groupby(["sku_id","product_name","unit_price"]).agg(
        n_pharmacies=      ("pharmacy_id", "count"),
        total_forecast_7d= ("forecast_7d", "sum"),
    ).reset_index()
    agg["total_forecast_7d"] = agg["total_forecast_7d"].round(0).astype(int)

    # Join distributor stock
    stock = (
        dist_stock_api[dist_stock_api["distributor_id"]==distributor_id]
        .sort_values("snapshot_date")
        .groupby("sku_id").last().reset_index()
        [["sku_id","current_stock","is_low_stock","is_out_of_stock"]]
    )
    result = agg.merge(stock, on="sku_id", how="left")
    result["stock_gap"]      = (result["total_forecast_7d"] - result["current_stock"]).clip(lower=0).astype(int)
    result["coverage_ratio"] = (result["current_stock"] / result["total_forecast_7d"].clip(lower=1)).round(3)

    dist_info  = dist_df_api[dist_df_api["dist_id"]==distributor_id].iloc[0]
    at_risk    = result[result["stock_gap"]>0]["product_name"].tolist()

    prompt = (
        f"Swasthya distributor. 2 sentences. No markdown.\n"
        f"Distributor: {dist_info["name"]}, {dist_info["city"]}\n"
        f"Expected: {result["total_forecast_7d"].sum():,} units | "
        f"At risk: {', '.join(at_risk) if at_risk else 'None'}\n"
        f"Lead time: {dist_info["lead_time_days"]}d"
    )
    ai_insight = gemini.generate_content(prompt).text.strip()

    return {
        "distributor_id":    distributor_id,
        "distributor_name":  dist_info["name"],
        "n_pharmacies":      len(assigned_ph),
        "total_expected_7d": int(result["total_forecast_7d"].sum()),
        "total_gap":         int(result["stock_gap"].sum()),
        "n_at_risk_skus":    int((result["stock_gap"]>0).sum()),
        "products":          result.to_dict("records"),
        "ai_insight":        ai_insight,
    }
'''
print(FASTAPI_CODE)
print('\n✅ Copy the above into ML/api/main.py')

---
## Feature 3 : Distributor Aggregation

### Files saved in feature3_outputs.zip:
```
aggregated_demand.csv            ← distributor-level 7d forecast
distributor_risk.csv             ← stock risk per distributor+SKU
pharmacy_forecasts.csv           ← pharmacy-level forecasts
distributor_insights.csv         ← Gemini briefings for all 5
dist_metadata.json               ← config for FastAPI
charts/distributor_overview.png
charts/distributor_dashboard_detail.png
```

### Complete ML/ folder (final state):
```
ML/
├── notebooks/
│   ├── Swasthya_Feature1_DemandForecasting.ipynb
│   ├── Swasthya_Feature2_AnomalyDetection.ipynb
│   └── Swasthya_Feature3_DistributorAggregation.ipynb
├── models/
│   ├── demand_forecaster.pkl        ← used by Feature 1 AND Feature 3
│   ├── anomaly_detector.pkl         ← Feature 2
│   ├── anomaly_scaler.pkl           ← Feature 2
│   ├── entity_stats.csv             ← Feature 2 entity profiles
│   ├── model_metadata.json          ← Feature 1 metrics
│   └── anomaly_metadata.json        ← Feature 2 metrics
├── outputs/
│   ├── aggregated_demand.csv        ← Feature 3
│   └── distributor_risk.csv         ← Feature 3
└── api/
    └── main.py                      ← /forecast
                                        /detect-anomaly
                                        /distributor-forecast
```


### All 3 features summary:
| Feature | Role | Model | Gemini role |
|---|---|---|---|
| 1 — Demand Forecasting | Pharmacy | LightGBM (trained) | Reorder recommendation |
| 2 — Anomaly Detection | Admin | Isolation Forest (trained) | Admin alert |
| 3 — Demand Aggregation | Distributor | Reuses Feature 1 | Supply briefing |